# Readmission for Healthcare Analysis

## Initial Exploration

This notebook loads the dataset and performs initial exploration to understand its structure, variables, and data quality.  

# Team Members

- Michael Ryan
- Roy Phelps - Data exploration, project setup, and data analytics

# Project Scope

Using a synthetic dataset, our goal is to determine under what conditions does a patient gets readmitted and predict if a patient will be readmitted.  

# Description of the Dataset

The datasets' fields include

- patient id
- admission date
- season
- age
- gender
- redion
- primary diagnosis
- comorbidites count **(condition that the patient has, ex: stroke, hypertension etc..)**
- length of stay
- treaement type
- medications count
- follow up visits prev. year
- insurance type
- discharge disposition
- readmission risk score
- label **(readmitted in the past 30 days binary, 1 = yes, 0 = no)**

In [8]:
# Imports
import pandas as pd
import numpy as np

# Uncomment for first install
#!pip install openpyxl

In [12]:
# Read in the dataset and view the first 5 entries
df = pd.read_csv("../data/raw/hospital_readmission_dataset.csv")
df.head()

,patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
0,P00001,2022-04-14,Spring,66,Male,South,Diabetes,5,6,Interventional,8,6,1,Medicare,Home Health,0.92,1
1,P00002,2021-09-19,Fall,55,Male,South,Diabetes,4,6,Interventional,6,4,3,Private,Home Health,0.88,1
2,P00003,2023-04-12,Spring,69,Female,West,Hypertension,6,8,Medical,9,6,2,Medicare,Skilled Nursing,0.97,1
3,P00004,2023-08-14,Summer,83,Male,South,Stroke,6,11,Medical,11,4,2,Medicare,Skilled Nursing,0.97,1
4,P00005,2021-11-05,Fall,54,Female,North,Stroke,4,10,Medical,6,2,1,Uninsured,Home Health,0.83,1


# Dataset Structure

We start by examining the size, structure, metrics, missing values of the dataset

In [34]:
# Dataset shape
df.shape

(8000, 17)

In [36]:
# Dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_id                 8000 non-null   object 
 1   admission_date             8000 non-null   object 
 2   season                     8000 non-null   object 
 3   age                        8000 non-null   int64  
 4   gender                     8000 non-null   object 
 5   region                     8000 non-null   object 
 6   primary_diagnosis          8000 non-null   object 
 7   comorbidities_count        8000 non-null   int64  
 8   length_of_stay             8000 non-null   int64  
 9   treatment_type             8000 non-null   object 
 10  medications_count          8000 non-null   int64  
 11  followup_visits_last_year  8000 non-null   int64  
 12  prev_readmissions          8000 non-null   int64  
 13  insurance_type             8000 non-null   objec

In [38]:
# Dataset metrics
df.describe()

,age,comorbidities_count,length_of_stay,medications_count,followup_visits_last_year,prev_readmissions,readmission_risk_score,label
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.00000,8000.000000,8000.000000
mean,57.411625,4.318500,7.789125,7.475750,3.638125,1.57250,0.776937,0.772875
std,16.724388,1.358046,1.930252,2.287917,1.630415,0.89196,0.219885,0.419000
min,18.000000,1.000000,3.000000,2.000000,0.000000,0.00000,0.070000,0.000000
25%,46.000000,3.000000,6.000000,6.000000,2.000000,1.00000,0.630000,1.000000
50%,57.000000,4.000000,8.000000,8.000000,4.000000,1.00000,0.860000,1.000000
75%,69.000000,5.000000,9.000000,9.000000,4.000000,2.00000,0.970000,1.000000
max,95.000000,10.000000,15.000000,18.000000,10.000000,5.00000,0.970000,1.000000


In [40]:
# Check missing values
df.isnull().sum()

patient_id                   0
admission_date               0
season                       0
age                          0
gender                       0
region                       0
primary_diagnosis            0
comorbidities_count          0
length_of_stay               0
treatment_type               0
medications_count            0
followup_visits_last_year    0
prev_readmissions            0
insurance_type               0
discharge_disposition        0
readmission_risk_score       0
label                        0
dtype: int64

In [42]:
# Check duplicate rows
df.duplicated().sum()

0

In [44]:
# Target variable distribution
df['label'].value_counts()
df['label'].value_counts(normalize=True) * 100

label
1    77.2875
0    22.7125
Name: proportion, dtype: float64

# Feature Selection Considerations

This variable appears to represent an overall assessment of a patient's likelihood of hospital readmission. Because the methodology used to calculate this score is not documented, there is a possibility that it incorporates information that may not be available at the time a prediction would normally be made. If so, including this feature during model training could introduce **data leakage**, resulting in overly optimistic model performance.

To evaluate the impact of this variable, two predictive models will be developed:

1. **Model A:** Includes all available features, including `readmission_risk_score`.
2. **Model B:** Excludes `readmission_risk_score` to determine how well the remaining patient demographic and clinical variables predict hospital readmission.

The performance of both models will be compared to assess whether the inclusion of `readmission_risk_score` provides legitimate predictive value or introduces information that would not be available in a real-world prediction scenario.

This comparison will also provide insight into the importance of feature selection and the potential effects of data leakage in machine learning models.

> **Note:** Since this dataset is synthetic, the origin of the `readmission_risk_score` is unknown. Rather than assuming the feature is appropriate or inappropriate for modeling, both approaches will be evaluated and compared.
During the initial exploration of the dataset, one feature was identified that warrants further investigation: **`readmission_risk_score`**.